# Run the GLM model

This notebook fits the project GLM on one packed h5 session. It keeps only a small neuron subset by default so the first run gives immediate output.

## 1. Setup

The GLM uses the binary visual stimulus trace as input and each neuron's `dff` trace as output. The fitted result is one stimulus kernel per neuron.

In [ ]:
from pathlib import Path
import os

repo = Path.cwd()
if repo.name == 'notebooks':
    repo = repo.parent
os.chdir(repo)

H5_PATH = repo / 'results_pack' / 'YH21SC' / 'V1_random_01' / 'V1_random_01.h5'
N_NEURONS = 20
KERNEL_WIN_MS = [-1500, 1500]

print(H5_PATH)
print('exists:', H5_PATH.exists())

## 2. Load one session

Dimensions used below: `dff = (n_neurons, n_times)`, `time = (n_times,)`, `vol_stim_vis = (n_vol_times,)`, and `stim_labels = (n_stim, 8)`.

In [ ]:
import h5py
import numpy as np

with h5py.File(H5_PATH, 'r') as f:
    labels = f['labels'][:N_NEURONS]
    dff = f['neural_trials']['dff'][:N_NEURONS]
    time = f['neural_trials']['time'][:]
    vol_time = f['neural_trials']['vol_time'][:]
    vol_stim_vis = f['neural_trials']['vol_stim_vis'][:]
    stim_labels = f['neural_trials']['stim_labels'][:]

print('labels:', labels.shape)
print('dff:', dff.shape)
print('time:', time.shape)
print('vol_time:', vol_time.shape)
print('vol_stim_vis:', vol_stim_vis.shape)
print('stim_labels:', stim_labels.shape)

## 3. Build the GLM time window

`run_glm_multi_sess` expects the kernel window as frame counts: `l_idx` frames before stimulus onset and `r_idx` frames after onset.

In [ ]:
dt = float(np.nanmedian(np.diff(time)))
l_idx = int(abs(KERNEL_WIN_MS[0]) / dt)
r_idx = int(abs(KERNEL_WIN_MS[1]) / dt)
kernel_time = np.arange(-l_idx, r_idx) * dt

print('frame interval:', round(dt, 3), 'ms')
print('l_idx:', l_idx, 'r_idx:', r_idx)
print('kernel_time:', kernel_time.shape, 'range:', (round(kernel_time[0], 1), round(kernel_time[-1], 1)))

## 4. Fit the GLM

The function interpolates `vol_stim_vis` onto imaging time, builds lagged stimulus regressors, and solves a ridge regression for each neuron.

In [ ]:
from modeling.generative import run_glm_multi_sess

kernel_all = run_glm_multi_sess(
    [dff], [time],
    [vol_time], [vol_stim_vis], [stim_labels],
    l_idx, r_idx)

glm = {'kernel_time': kernel_time, 'kernel_all': kernel_all}
print('kernel_all:', glm['kernel_all'].shape)
print('kernel_time:', glm['kernel_time'].shape)

## 5. Print immediate results

A quick first read is the peak kernel amplitude and when it occurs for each neuron.

In [ ]:
peak_idx = np.nanargmax(np.abs(kernel_all), axis=1)
peak_amp = kernel_all[np.arange(kernel_all.shape[0]), peak_idx]
peak_time = kernel_time[peak_idx]

print('neuron  label  peak_time_ms  peak_amp')
for i in range(min(10, len(peak_amp))):
    print(f'{i:6d} {labels[i]:6d} {peak_time[i]:12.1f} {peak_amp[i]:9.4f}')

print('\nmean abs peak:', float(np.nanmean(np.abs(peak_amp))))

## 6. Plot kernels

The first panel shows individual kernels. The second panel shows the mean kernel with SEM across the selected neurons.

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(11, 3), constrained_layout=True)

for k in kernel_all[:min(10, kernel_all.shape[0])]:
    axs[0].plot(kernel_time, k, lw=1, alpha=0.8)
axs[0].axvline(0, color='k', lw=1, ls='--')
axs[0].set_title('Individual GLM kernels')
axs[0].set_xlabel('time from stimulus onset (ms)')
axs[0].set_ylabel('kernel weight')

m = np.nanmean(kernel_all, axis=0)
s = np.nanstd(kernel_all, axis=0) / np.sqrt(kernel_all.shape[0])
axs[1].plot(kernel_time, m, color='black', lw=2)
axs[1].fill_between(kernel_time, m-s, m+s, color='black', alpha=0.2, lw=0)
axs[1].axvline(0, color='k', lw=1, ls='--')
axs[1].set_title('Mean GLM kernel')
axs[1].set_xlabel('time from stimulus onset (ms)')
axs[1].set_ylabel('kernel weight')

plt.show()

## 7. What this means

`kernel_all[i, :]` is the estimated visual stimulus response kernel for neuron `i`. Positive weights mean the visual stimulus predicts higher dF/F at that lag; negative weights mean it predicts lower dF/F. The full figure code stores the same structure as `glm['kernel_all']` and passes it to the TRF model.